In [1]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [2]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

# Changed this line slightly to always overwrite files so we don't get prompted every time
!unzip -o book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2026-11-22 11:45:04--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.2.33, 172.67.70.149, 104.26.3.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.2.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip’

book-crossings.zip  100%[===================>]  24.88M  --.-KB/s    in 0.1s    

2026-11-22 11:45:04 (173 MB/s) - ‘book-crossings.zip’ saved [26085508/26085508]

Archive:  book-crossings.zip
  inflating: BX-Book-Ratings.csv     
  inflating: BX-Books.csv            
  inflating: BX-Users.csv            


In [3]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [4]:
# add your code here - consider creating a new cell for each section of code
df_books

,isbn,title,author
0,0195153448,Classical Mythology,Mark P. O. Morford
1,0002005018,Clara Callan,Richard Bruce Wright
2,0060973129,Decision in Normandy,Carlo D'Este
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata
4,0393045218,The Mummies of Urumchi,E. J. W. Barber
...,...,...,...
271374,0440400988,There's a Bat in Bunk Five,Paula Danziger
271375,0525447644,From One to One Hundred,Teri Sloat
271376,006008667X,Lily Dale : The True Story of the Town that Ta...,Christine Wicker
271377,0192126040,Republic (World's Classics),Plato


In [5]:
df_ratings

,user,isbn,rating
0,276725,034545104X,0.0
1,276726,0155061224,5.0
2,276727,0446520802,0.0
3,276729,052165615X,3.0
4,276729,0521795028,6.0
...,...,...,...
1149775,276704,1563526298,9.0
1149776,276706,0679447156,0.0
1149777,276709,0515107662,10.0
1149778,276721,0590442449,10.0


# Filter out rows that have incomplete data

In [6]:
# What rows have any missing values?
df_ratings.isnull().sum()

,0
user,0
isbn,0
rating,0


The ratings dataset is free from null values

In [7]:
# What book rows have missing values?
df_books.isnull().sum()

,0
isbn,0
title,0
author,2


In [8]:
# Which book is missing an author?
df_books[df_books['author'].isnull()]

,isbn,title,author
118038,0751352497,A+ Quiz Masters:01 Earth,NaN
187700,9627982032,The Credit Suisse Guide to Managing Your Perso...,NaN


In [9]:
# Drop the rows with missing values from the book dataframe:
df_books.dropna(inplace=True)
df_books.isnull().sum()

,0
isbn,0
title,0
author,0


## Data cleanup

1. Remove any users that have less than 200 ratings
2. Remove any book with less than 100 ratings

### Remove users with less than 200 ratings

In [10]:
user_ratings = df_ratings['user'].value_counts()
print("Number of users before cleanup: ", len(user_ratings))
print("Number of rows before cleanup: ", len(df_ratings))


Number of users before cleanup:  105283
Number of rows before cleanup:  1149780


In [11]:
ratings = df_ratings['user'].value_counts()
ratings.sort_values(ascending=False).head()

,count
user,
11676,13602
198711,7550
153662,6109
98391,5891
35859,5850


In [12]:
# number of users with less than 200 rated books
total_users = len(ratings)
users_less_200 = len(ratings[ratings < 200])
delta_users = total_users - users_less_200
print("Total number of users:", total_users)
print("Users with less than 200 rated books:", users_less_200)
print("Users with 200 or more rated books:", delta_users)

Total number of users: 105283
Users with less than 200 rated books: 104378
Users with 200 or more rated books: 905


In [13]:
df_ratings.groupby("user")

In [14]:
delta_users

905

In [15]:
# Filter out users with less than 200 ratings
df_ratings_rm = df_ratings.groupby("user").filter(
    lambda x: len(x) >= 200)
user_counts_post_user_cleanup  = df_ratings_rm['user'].value_counts()
len_user_counts_post_user_cleanup = len(user_counts_post_user_cleanup)
n_rows_after_user_cleanup = len(df_ratings_rm)

print("# of users after cleanup:", len_user_counts_post_user_cleanup)
print("# of rows after cleanup:", n_rows_after_user_cleanup,"\n")


# There should be 527,556 ratings left
assert n_rows_after_user_cleanup == 527_556
# Assert that there are 905 users left, as calculated earlier
assert(len_user_counts_post_user_cleanup == delta_users)
user_counts_post_user_cleanup

# of users after cleanup: 905
# of rows after cleanup: 527556 



,count
user,
11676,13602
198711,7550
153662,6109
98391,5891
35859,5850
...,...
26883,200
225595,200
83671,200


In [16]:
df_ratings_rm

,user,isbn,rating
1456,277427,002542730X,10.0
1457,277427,0026217457,0.0
1458,277427,003008685X,8.0
1459,277427,0030615321,0.0
1460,277427,0060002050,0.0
...,...,...,...
1147612,275970,3829021860,0.0
1147613,275970,4770019572,0.0
1147614,275970,896086097,0.0
1147615,275970,9626340762,8.0


### Also remove books with less than 100 ratings from the same dataframe

It's important to realize we must select the books from the original dataframe, since many book ratings have already been filtered out.

In [17]:
# Metrics for removing books with less than 100 ratings

# we have to use the original df_ratings to pass the challenge, because
# otherwise we could be filtering out books that originally had more than
# a 100 ratings
ratings = df_ratings['isbn'].value_counts()
over_100_ratings = ratings[ratings >= 100]
under_100_ratings = ratings[ratings < 100]
print("# of rated books:", len(ratings))
print("# of books with at least 100 ratings:", len(over_100_ratings))
print("# number of books with less than 100 ratings", len(under_100_ratings))

# of rated books: 340556
# of books with at least 100 ratings: 731
# number of books with less than 100 ratings 339825


In [18]:
over_100_ratings

,count
isbn,
0971880107,2502
0316666343,1295
0385504209,883
0060928336,732
0312195516,723
...,...
067100042X,100
0425151867,100
0425136981,100


In [19]:
df_ratings_rm

,user,isbn,rating
1456,277427,002542730X,10.0
1457,277427,0026217457,0.0
1458,277427,003008685X,8.0
1459,277427,0030615321,0.0
1460,277427,0060002050,0.0
...,...,...,...
1147612,275970,3829021860,0.0
1147613,275970,4770019572,0.0
1147614,275970,896086097,0.0
1147615,275970,9626340762,8.0


In [20]:
under_100_ratings

,count
isbn,
0671727583,99
0330375253,99
0553274503,99
0553280414,99
0553569058,99
...,...
0671883917,1
0743257502,1
0767409752,1


In [21]:
# Filter out books with less than 100 ratings
df_ratings_rm = df_ratings_rm[
  ~df_ratings_rm['isbn'].isin(under_100_ratings.index)
]
df_ratings_rm.shape
print("# of books left in the dataset:", len(df_ratings_rm.isbn.value_counts()))
print("Rows left after removing books:", len(df_ratings_rm))

# of books left in the dataset: 731
Rows left after removing books: 49781


In [22]:
df_ratings_rm['isbn'].value_counts()

,count
isbn,
0971880107,365
0316666343,272
0060928336,221
0440214041,218
0385504209,217
...,...
039592720X,23
076790592X,23
0684833395,22


In [23]:
df_ratings_rm['user'].value_counts()

,count
user,
11676,660
35859,348
16795,297
76352,291
23768,259
...,...
144131,1
203820,1
153621,1


In [24]:
# Sanity check:
# These should exist, since we're being graded on them being nearest neighbors
books = ["Where the Heart Is (Oprah's Book Club (Paperback))",
        "I'll Be Seeing You",
        "The Weight of Water",
        "The Surgeon",
        "I Know This Much Is True"]

for book in books:
  print(df_ratings.isbn.isin(df_books[df_books.title == book].isbn).sum())

585
145
117
175
276


In [25]:
df_ratings_rm.shape

(49781, 3)

In [26]:
df_ratings_rm.user.value_counts()

,count
user,
11676,660
35859,348
16795,297
76352,291
23768,259
...,...
144131,1
203820,1
153621,1


# Prepare the dataset for KNN
1. Pivot the ratings df so that it's indexed by user,
has the ISBN in the columns, and the values are the users' ratings.
2. Any unrated books by the user gets a value of 0
3. Transpose the table so that user index is in the columns and the ratings are in the rows
4. Exchange the ISBN index in the ratings dataset for the book title by doing a join with the books df.

In [27]:
knn_df = df_ratings_rm.pivot_table(index=['user'],columns=['isbn'],values='rating').fillna(0)
knn_df.head()

isbn,002542730X,0060008032,0060096195,006016848X,0060173289,0060175400,006019491X,0060199652,0060391626,0060392452,...,1558744630,1558745157,1559029838,1573225517,1573225789,1573227331,1573229326,1573229571,1592400876,1878424319
user,,,,,,,,,,,,,,,,,,,,,
254,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2276,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2766,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2977,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3363,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [28]:
knn_df = knn_df.T
knn_df.head()

user,254,2276,2766,2977,3363,4017,4385,6242,6251,6323,...,274004,274061,274301,274308,274808,275970,277427,277478,277639,278418
isbn,,,,,,,,,,,,,,,,,,,,,
002542730X,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0
0060008032,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0060096195,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
006016848X,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0060173289,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Exchange the ISBN index for the books' titles since we're using that for now.

In [29]:
knn_df.join(df_books.set_index('isbn'))['title']


,title
isbn,
002542730X,Politically Correct Bedtime Stories: Modern Ta...
0060008032,Angels
0060096195,The Boy Next Door
006016848X,"Men Are from Mars, Women Are from Venus: A Pra..."
0060173289,Divine Secrets of the Ya-Ya Sisterhood : A Novel
...,...
1573227331,About a Boy
1573229326,How to Be Good
1573229571,About a Boy (Movie Tie-In)


In [30]:
index = knn_df.join(df_books.set_index('isbn'))['title']
index

,title
isbn,
002542730X,Politically Correct Bedtime Stories: Modern Ta...
0060008032,Angels
0060096195,The Boy Next Door
006016848X,"Men Are from Mars, Women Are from Venus: A Pra..."
0060173289,Divine Secrets of the Ya-Ya Sisterhood : A Novel
...,...
1573227331,About a Boy
1573229326,How to Be Good
1573229571,About a Boy (Movie Tie-In)


In [31]:
knn_df.index = index
knn_df = knn_df.sort_index()
knn_df.head()

user,254,2276,2766,2977,3363,4017,4385,6242,6251,6323,...,274004,274061,274301,274308,274808,275970,277427,277478,277639,278418
title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2nd Chance,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2nd Chance,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [32]:
knn_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 731 entries, 1984 to nan
Columns: 888 entries, 254 to 278418
dtypes: float32(888)
memory usage: 2.5+ MB


In [33]:
knn_df.tail()

user,254,2276,2766,2977,3363,4017,4385,6242,6251,6323,...,274004,274061,274301,274308,274808,275970,277427,277478,277639,278418
title,,,,,,,,,,,,,,,,,,,,,
"\O\"" Is for Outlaw""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
NaN,8.0,0.0,0.0,0.0,0.0,9.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Build the model

Use the K-nearest neighbors algorithm to create the reccomendations.

Our table is now an array where the books are points in an n-dimensional hyperspace.

In [34]:
knn_df.values

array([[9., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [8., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(731, 888), dtype=float32)

In [35]:
# 1. Build the model
# Maybe cosine used to be default?
# We have to set it explicitly or the test distances will not be right...
model = NearestNeighbors(metric='cosine')
model.fit(knn_df.values)

NearestNeighbors(metric='cosine')

In [36]:
title = '1984'
distance, indices = model.kneighbors([knn_df.loc[title].values], n_neighbors=6)

print(distance)
print(indices)

[[0.         0.75083876 0.7530067  0.7618247  0.76593226 0.77405274]]
[[  0 548 487  56 497 436]]


In [37]:
for e in knn_df.iloc[indices[0]].index.values:
  print(e)

1984
The Handmaid's Tale
The Bonesetter's Daughter
Animal Farm
The Catcher in the Rye
Slaughterhouse Five or the Children's Crusade: A Duty Dance With Death


In [38]:
pd.DataFrame({
    'title'   : knn_df.iloc[indices[0]].index.values,
    'distance': distance[0]
}) \
.sort_values(by='distance', ascending=False)

,title,distance
5,Slaughterhouse Five or the Children's Crusade:...,0.774053
4,The Catcher in the Rye,0.765932
3,Animal Farm,0.761825
2,The Bonesetter's Daughter,0.753007
1,The Handmaid's Tale,0.750839
0,1984,0.000000


☝ Seems like a reasonable result☝
  
especially since Animal Farm is also there


In [39]:
# function to return recommended books - this will be tested
def get_recommends(title = ""):
  print(title)
  try:
    book = knn_df.loc[title]
  except KeyError as e:
    print('The given book', e, 'does not exist')
    return

  distances, indices = model.kneighbors([book.values], n_neighbors=6)

  recommended_books = pd.DataFrame({
      'title'   : knn_df.iloc[indices[0]].index.values,
      'distance': distances[0]
    }) \
    .sort_values(by='distance', ascending=False) \
    .head(5).values

  return [title, recommended_books]

In [40]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

Where the Heart Is (Oprah's Book Club (Paperback))
["Where the Heart Is (Oprah's Book Club (Paperback))", array([["I'll Be Seeing You", 0.8016210794448853],
       ['The Weight of Water', 0.7708583474159241],
       ['The Surgeon', 0.7699410915374756],
       ['I Know This Much Is True', 0.7677075266838074],
       ['The Lovely Bones: A Novel', 0.7234864234924316]], dtype=object)]
Where the Heart Is (Oprah's Book Club (Paperback))
You passed the challenge! 🎉🎉🎉🎉🎉
